# 01 · Define & Explore — choose the target, success criteria, controls, and the ML framing

**Standard slot:** *define & explore.* **For Project 25 (the capstone) this means:** make the two
decisions the whole campaign rests on — (1) **which real target** you will design against (any tier),
and (2) how you will **close the loop** by learning, from the cohort's accumulated design→outcome
data, which features actually predict success. You also state measurable success criteria + controls
and run a deterministic **mock / EXAMPLE_DATA** hello-world (D0).

> **This is the INTEGRATIVE capstone.** It reuses a full campaign workflow (here we use a **binder**
> as the worked example — but the *design_type is whichever you and your advisor choose*: binder,
> antibody, enzyme, monomer...) **and** adds the cohort-wide **ML success predictor** that is the
> project's signature deliverable.

> **RESPONSIBLE RESEARCH — read first.** Your **advisor MUST approve your chosen target against the
> `MASTER_BLUEPRINT.md §7` policy BEFORE you start the design campaign (P2).** Default any ambiguous
> target to a neutralizing / diagnostic / inhibitory / industrial framing. No out-of-scope targets
> (nothing that enhances pathogen transmissibility/virulence, toxins, screening evasion, or is meant
> to cause harm). See `README.md` / `MANUAL.md §8`.

Run `00_setup.ipynb` first in this session.

## The four non-negotiable messages (the capstone must embody all four)

1. **A computational design is a hypothesis, not a result.** A low `pae_interaction` / `scrmsd` is
   *confidence*, not measured binding or function.
2. **Diversity before filtering.** Generate many (100s–1000s), filter aggressively, never polish one.
3. **Controls are mandatory** — even in the *plan* (positive, scrambled-interface/dead-mutant
   negative, unrelated-protein negative).
4. **Report the hit rate, not the cherry.** Include the failures; report N pass / N generated.

The capstone adds a fifth, data-driven idea: **learn from the whole cohort which filters actually
work.** No single in-silico metric perfectly separates true from false hits — so we *train a model*
on accumulated design→outcome data and ask, honestly, whether it beats the field's single-metric
cutoffs.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Choose + justify your target (responsible-research check)

Fill in the block below for **your** advisor-approved target. The worked example is a **binder vs a
checkpoint protein** (the Project 06 PD-L1 pattern), but your `DESIGN_TYPE` is whatever you chose.

A defensible choice states: the protein, the *function* you want (neutralize / inhibit / diagnose /
catalyze an industrial reaction), the in-scope framing, and the structural data you will design
against. **Do not proceed to P2 until your advisor signs off the responsible-research check.**

In [ ]:
# ---- EDIT THIS BLOCK FOR YOUR TARGET (advisor-approved) ----
TARGET_NAME   = "EXAMPLE_TARGET (worked example: a checkpoint protein, binder design)"
DESIGN_TYPE   = "binder"     # binder | antibody | enzyme | monomer  (YOUR choice; drives the filter cutoffs)
FRAMING       = "inhibitory / therapeutic-diagnostic (in scope under MASTER_BLUEPRINT §7)"
STRUCTURE_SRC = "RCSB accession(s) you verify in Week 1 (see data/README.md)"
ADVISOR_APPROVED = False     # <-- set True only AFTER your advisor signs the §7 responsible-research check

assert DESIGN_TYPE in {"binder", "antibody", "enzyme", "monomer"}, "pick a supported design_type"
print("target      :", TARGET_NAME)
print("design_type :", DESIGN_TYPE, " (drives the shared-filter cutoffs in notebook 03)")
print("framing     :", FRAMING)
print("advisor responsible-research approval recorded:", ADVISOR_APPROVED)
if not ADVISOR_APPROVED:
    print("\n>>> STOP before P2 (the design campaign) until ADVISOR_APPROVED is True. <<<")

## 2 · Success criteria + controls (write them down NOW, not after)

State measurable criteria up front so you cannot move the goalposts later. Example for a binder:
"≥ N designs passing all binder layers with `pae_interaction ≤ 10` and `rosetta_dG ≤ −30`, with a
PD-1-footprint overlap ≥ 0.5"; for an enzyme: "≥ N designs with catalytic-geometry RMSD ≤ 0.5 Å."

Controls you commit to (used in the validation plan, notebook 05):
- **Positive:** a known-good binder/enzyme/natural protein (confirms the assay works).
- **Negative (scrambled-interface / catalytic dead-mutant):** your *own* top design, broken — it must
  lose activity. The cleanest specificity control.
- **Unrelated-protein negative.**

In [ ]:
SUCCESS_CRITERIA = {
    "min_designs_all_layers": 5,          # >=5 designs passing every filter layer (EDIT for your target)
    "key_metric": "pae_interaction <= 10" if DESIGN_TYPE in ("binder", "antibody") else "catalytic_geom_rmsd <= 0.5",
    "novelty": "report TM-score to PDB (novel if < 0.5)",
}
CONTROLS = ["positive: known-good binder/enzyme/natural protein",
            "negative: scrambled-interface / catalytic dead-mutant (your own top design, broken)",
            "negative: unrelated protein of similar size"]
print("success criteria:")
for k, v in SUCCESS_CRITERIA.items():
    print(f"  {k}: {v}")
print("\ncontrols (committed up front):")
for c in CONTROLS:
    print("  -", c)

## 3 · The ML framing — the capstone's signature

The campaign produces designs with in-silico metrics. The cohort (Projects 01–24) produced **many
more**, some with experimental labels. The capstone's distinctive question:

> *Given the cohort's accumulated `features → outcome` data, can we learn a predictor that triages
> designs better than the field's single-metric cutoffs (scRMSD<2, pae_interaction<10, ...)?*

`scripts/ml_predictor.py` is import-safe and deterministic. For teaching with no real cohort yet, it
builds a clearly-labeled **`EXAMPLE_DATA`** synthetic cohort (deterministic seed) with a *planted,
imperfect* feature→outcome structure — so this notebook runs anywhere. **Every synthetic row is
flagged `EXAMPLE_DATA`; never present these as real outcomes.**

In [ ]:
import ml_predictor as ml

# Hello-world: build a tiny EXAMPLE_DATA cohort slice and peek at the schema.
demo = ml.build_cohort_table(n_per_target=20, seed=0)
print("EXAMPLE_DATA cohort slice:", demo.shape)
print("feature columns the predictor learns from:", ml.FEATURE_COLUMNS)
print("\nfield-standard single-metric cutoffs (the baselines to beat):")
for k, (op, thr) in ml.STANDARD_CUTOFFS.items():
    print(f"  {k} {op} {thr}")
print("\nsuccess-rate in this EXAMPLE_DATA slice (synthetic!):", round(demo["success"].mean(), 3))
demo.head(4)[["design_id", "design_type", "scrmsd", "pae_interaction", "plddt", "source", "success"]]

## 4 · Mock hello-world: a tiny binder mini-run (worked example)

The campaign workflow (notebook 02) uses the binder-family pattern as the worked example. Here we run
the deterministic **mock** backend so the plumbing executes with no GPU. If your `DESIGN_TYPE` is an
enzyme/antibody, you will swap in that family's generator in notebook 02 — the *integration pattern*
(generate → score → cohort table → filter → ML) is identical. **Mock numbers are `SYNTHETIC`.**

In [ ]:
import campaign_tools as ct

TARGET = "EXAMPLE_TARGET"
HOTSPOTS = ct.parse_hotspots("A56,A66,A115")   # EXAMPLE — replace with residues you derive from your target
designs = ct.generate_designs(TARGET, HOTSPOTS, n=4, design_type=DESIGN_TYPE, tool="mock")
ct.score_designs(designs, tool="mock")
d = designs[0]
print("example design:", d.design_id, "| len =", d.length, "aa")
print("  scrmsd =", d.scrmsd, " pae_interaction =", d.pae_interaction,
      " plddt =", d.plddt, " (SYNTHETIC)")
print("  synthetic flag:", d.synthetic, "->", d.notes[0])
print("\nSwitch tool='mock' -> the real backend (BindCraft/RFdiffusion/RFantibody/RFdiffusion2) on Colab (A100). See MANUAL.md §2.")

## D0 checklist
- [ ] **Target chosen + justified**; `DESIGN_TYPE` set; structural source noted (verify accessions Week 1).
- [ ] **Advisor responsible-research approval (§7) recorded** — required before P2.
- [ ] Measurable **success criteria** + **controls** written down up front.
- [ ] ML framing understood; `EXAMPLE_DATA` cohort slice inspected (flagged synthetic).
- [ ] Reproduced the mock mini-run (worked-example binder) with metrics printed and flagged SYNTHETIC.
- [ ] `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the full design campaign + assemble the cohort feature table.